# PathLens-GNN — staged Kaggle run

`STAGE` defaults to `smoke`. A Save & Run All therefore cannot tune or open the sealed test unless the operator explicitly changes the stage. Each stage emits a resumable ZIP under `/kaggle/working`.

In [ ]:
STAGE = "smoke"  # smoke | registered | tuning | confirmation | freeze | final
GIT_REF = "research/ranking-loss"  # Campaign v2 ranking-loss branch.
RESUME_ARCHIVE = None  # Example: /kaggle/input/pathlens-output/pathlens-stage-output.zip
SESSION_BUDGET_SECONDS = 9 * 60 * 60  # Leave time to archive outputs before Kaggle stops.
FINAL_TEST_TOKEN = ""  # Set to OPEN_SEALED_TEST_ONCE only for the authorized final stage.

ALLOWED_STAGES = {"smoke", "registered", "tuning", "confirmation", "freeze", "final"}
if STAGE not in ALLOWED_STAGES:
    raise ValueError(f"Unsupported stage: {STAGE}")

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys

REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
OUTPUT = pathlib.Path("/kaggle/working/output")
PROCESSED = pathlib.Path("/kaggle/working/data/processed")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
repository_source = str(REPO / "src")
if repository_source not in sys.path:
    sys.path.insert(0, repository_source)

In [ ]:
from pathlens_gnn.training.kaggle import restore_output_archive

if RESUME_ARCHIVE is not None:
    restore_output_archive(RESUME_ARCHIVE, OUTPUT)
    print(f"Restored prior output from {RESUME_ARCHIVE}")

In [ ]:
from datetime import UTC, datetime

environment_stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
environment_path = OUTPUT / f"environment-{STAGE}-{environment_stamp}.json"
subprocess.run(
    [
        "python",
        "scripts/capture_environment.py",
        "--repository",
        str(REPO),
        "--output",
        str(environment_path),
    ],
    check=True,
)
environment = json.loads(environment_path.read_text())
if not environment["torch"]["cuda_available"]:
    raise RuntimeError("Kaggle GPU is not active")
environment

In [ ]:
import gzip
import urllib.request

raw_dir = pathlib.Path("/kaggle/working/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
source = raw_dir / "biosnap.tsv"
if not source.exists():
    archive = raw_dir / "biosnap.tsv.gz"
    urllib.request.urlretrieve(
        "https://snap.stanford.edu/biodata/datasets/10002/files/ChG-Miner_miner-chem-gene.tsv.gz",
        archive,
    )
    with gzip.open(archive, "rb") as inp, source.open("wb") as out:
        shutil.copyfileobj(inp, out)
if not (PROCESSED / "manifest.json").exists():
    subprocess.run(
        [
            "pathlens",
            "prepare-data",
            "--source",
            str(source),
            "--output",
            str(PROCESSED),
            "--seed",
            "41",
        ],
        check=True,
    )
manifest = json.loads((PROCESSED / "manifest.json").read_text())
if (
    manifest.get("dataset_version") != "biosnap-dti-canonical-v2"
    or manifest.get("seed") != 41
):
    raise RuntimeError(
        "Unexpected processed split: "
        f"{manifest.get('dataset_version')} seed={manifest.get('seed')}"
    )
expected = {"edges": 15138, "drugs": 5017, "proteins": 2324, "entities": 7341}
for key, value in expected.items():
    if manifest["counts"][key] != value:
        raise RuntimeError(f"Unexpected BioSNAP {key}: {manifest['counts'][key]}")
manifest["counts"]

In [ ]:
if STAGE == "smoke":
    subprocess.run(
        [
            sys.executable,
            "-u",
            "scripts/train.py",
            "--processed",
            str(PROCESSED),
            "--output",
            str(OUTPUT / "smoke"),
            "--config",
            "configs/model/smoke.yaml",
        ],
        check=True,
    )
elif STAGE == "registered":
    subprocess.run(
        [
            sys.executable,
            "-u",
            "scripts/run_registered_experiments.py",
            "--processed",
            str(PROCESSED),
            "--output",
            str(OUTPUT / "registered"),
        ],
        check=True,
    )
elif STAGE == "tuning":
    subprocess.run(
        [
            sys.executable,
            "-u",
            "scripts/run_tuning.py",
            "--processed",
            str(PROCESSED),
            "--output",
            str(OUTPUT / "tuning"),
            "--session-seconds",
            str(SESSION_BUDGET_SECONDS),
        ],
        check=True,
    )

In [ ]:
if STAGE == "confirmation":
    leaderboard = OUTPUT / "tuning/leaderboard.json"
    if not leaderboard.exists():
        raise RuntimeError("Restore a completed tuning output before confirmation")
    subprocess.run(
        [
            "python",
            "scripts/run_confirmation.py",
            "--processed",
            str(PROCESSED),
            "--leaderboard",
            str(leaderboard),
            "--output",
            str(OUTPUT / "confirmation"),
        ],
        check=True,
    )

In [ ]:
confirmation_path = OUTPUT / "confirmation/confirmation.json"
if STAGE == "freeze":
    if not confirmation_path.exists():
        raise RuntimeError("Restore confirmation output before freezing a model")
    confirmation = json.loads(confirmation_path.read_text())
    chosen = confirmation["selected"]["seed_13_checkpoint"]
    decision = (
        "Best preregistered mean validation result after the fixed compute budget; "
        "seed 13 is the preregistered evaluation checkpoint"
    )
    subprocess.run(
        [
            "python",
            "scripts/freeze_model.py",
            "--checkpoint",
            chosen,
            "--output",
            str(OUTPUT / "model-freeze.json"),
            "--decision",
            decision,
        ],
        check=True,
    )

In [ ]:
if STAGE == "final":
    final_output = OUTPUT / "final-evaluation.json"
    from pathlens_gnn.training.kaggle import authorize_final_evaluation

    authorize_final_evaluation(FINAL_TEST_TOKEN, final_output)
    freeze_record = OUTPUT / "model-freeze.json"
    if not freeze_record.exists():
        raise RuntimeError("Restore the frozen model output before final evaluation")
    freeze = json.loads(freeze_record.read_text())
    subprocess.run(
        [
            "python",
            "scripts/evaluate_checkpoint.py",
            "--processed",
            str(PROCESSED),
            "--checkpoint",
            freeze["checkpoint"],
            "--freeze-record",
            str(freeze_record),
            "--output",
            str(final_output),
            "--confirm-sealed-test",
        ],
        check=True,
    )

In [ ]:
from datetime import UTC, datetime

stage_record = {
    "stage": STAGE,
    "completed_at": datetime.now(UTC).isoformat(),
    "git_commit": subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
    ).stdout.strip(),
    "dataset_sha256": manifest["source"]["sha256"],
}
(OUTPUT / f"stage-{STAGE}.json").write_text(json.dumps(stage_record, indent=2))
shutil.make_archive("/kaggle/working/pathlens-stage-output", "zip", OUTPUT)
print(json.dumps(stage_record, indent=2))
print("Download or attach /kaggle/working/pathlens-stage-output.zip before the next stage.")